# Plant Doctor AI — Experiment C: Conditional DCGAN

Train a class-conditional DCGAN at 128x128 RGB using the training split only.
The validation and test splits are never used for GAN training.

Stop-loss: inspect generated samples before using them for classifier augmentation.

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

ZIP_PATH = Path('/content/drive/MyDrive/plant_doctor_dataset.zip')
DATASET_ROOT = Path('/content/plant_doctor_dataset')
REPO_DIR = Path('/content/Plant-Doctor-AI')
GAN_OUTPUT = Path('/content/drive/MyDrive/Plant-Doctor-AI/results/conditional_dcgan')

print('ZIP exists:', ZIP_PATH.exists())
print('Dataset extracted:', DATASET_ROOT.exists())
print('Repo exists:', REPO_DIR.exists())


In [ ]:
import zipfile, shutil

if not REPO_DIR.exists():
    !git clone https://github.com/atharavakadam21-crypto/Plant-Doctor-AI.git /content/Plant-Doctor-AI
%cd /content/Plant-Doctor-AI
!git pull origin main

if not all((DATASET_ROOT / s).is_dir() for s in ['train', 'validation', 'test']):
    if not ZIP_PATH.exists():
        raise FileNotFoundError(f'Dataset ZIP not found: {ZIP_PATH}')
    if DATASET_ROOT.exists():
        shutil.rmtree(DATASET_ROOT)
    DATASET_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        for member in z.infolist():
            name = member.filename.replace('\\', '/').lstrip('/')
            if not name:
                continue
            target = DATASET_ROOT / name
            if member.is_dir() or name.endswith('/'):
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with z.open(member) as src, open(target, 'wb') as dst:
                    shutil.copyfileobj(src, dst)

for split in ['train', 'validation', 'test']:
    p = DATASET_ROOT / split
    count = sum(1 for x in p.rglob('*') if x.is_file())
    print(f'{split}: {count}')

assert all((DATASET_ROOT / s).is_dir() for s in ['train', 'validation', 'test'])
print('✅ Dataset ready')

## Day 1 training

Run the GAN once and inspect the saved sample grids. Training uses only `train/`.

In [ ]:
!python -m gan.train_conditional_dcgan \
    --dataset-root /content/plant_doctor_dataset \
    --output-dir '/content/drive/MyDrive/Plant-Doctor-AI/results/conditional_dcgan' \
    --epochs 20 \
    --batch-size 64 \
    --num-workers 2 \
    --latent-dim 128 \
    --base-channels 64 \
    --lr 2e-4 \
    --samples-per-class 8 \
    --seed 42 \
    --device cuda

## Inspect the last generated grid

Do not use GAN images for classifier augmentation until the generated samples are visually inspected.

In [ ]:
from pathlib import Path
from IPython.display import display, Image

sample_dir = Path('/content/drive/MyDrive/Plant-Doctor-AI/results/conditional_dcgan/samples')
samples = sorted(sample_dir.glob('epoch_*.png'))
print('Sample grids:', len(samples))
if samples:
    print('Latest:', samples[-1])
    display(Image(filename=str(samples[-1])))